# Simulation Plotting Tools

## Import Data

In [4]:
import numpy as np
import ROOT
import pandas as pd
import random
import matplotlib.pyplot as plt

In [5]:
class Particle:
    def __init__(self, track_id, init_pos, init_mom, actual_hits, kalman_hits):
        self.track_id = track_id
        self.init_pos = init_pos
        self.init_mom = init_mom
        self.actual_hits = actual_hits
        self.kalman_hits = kalman_hits


class LayerHit:
    def __init__(self, hit_pos, hit_mom):
        self.hit_pos = hit_pos
        self.hit_mom = hit_mom

In [6]:
filepath = "output/"

# Get the actual and kalman fitted data from the ROOT file
ROOT.gSystem.Load("output/libDriftChamberlib.so")
with ROOT.TFile.Open("output/kalman_output.root", "READ") as file:
    if not file or file.IsZombie():
        print("Error opening file")

    tree = file.Get("Particles")

    particles = []

    for entry in tree:
        for particle in entry.Particles:
            track_id = particle.getID()
            init_pos = (particle.getPosition().X(), particle.getPosition().Y(), particle.getPosition().Z())
            init_mom = (particle.getMomentum().X(), particle.getMomentum().Y(), particle.getMomentum().Z())
            
            actual_hits = []
            kalman_hits = []

            for hit in particle.getActualHits():
                pos = (hit.getEntryPosition().X(), hit.getEntryPosition().Y(), hit.getEntryPosition().Z())
                mom = (hit.getEntryMomentum().X(), hit.getEntryMomentum().Y(), hit.getEntryMomentum().Z())

                layer_hit = LayerHit(pos, mom)
                actual_hits.append(layer_hit)
            
            for hit in particle.getKalmanHits():
                pos = (hit.getPosition().X(), hit.getPosition().Y(), hit.getPosition().Z())
                mom = (hit.getMomentum().X(), hit.getMomentum().Y(), hit.getMomentum().Z())

                layer_hit = LayerHit(pos, mom)
                kalman_hits.append(layer_hit)

            particle_hit = Particle(track_id, init_pos, init_mom, actual_hits, kalman_hits)
            particles.append(particle_hit)


# Load layer configuration and sample event row data
layer_df = pd.read_csv(filepath+'layer_radius.csv')

# Load cluster data
clusters_df = pd.read_csv(filepath+'cluster_info.csv')
clusters_df["hit_time"] = clusters_df["gen_time"] + clusters_df["drift_time"]

## Trajectory comparison

In [8]:
def get_accuracy(kalman_data, actual_data, ax_idx, ret_type):
    axes = ["x", "y", "z"]
    ax_key = axes[ax_idx]
    
    kax = kalman_data[ax_idx]
    avg_diff = 0

    p1, p2 = 0, 0
    
    for i in range(100):
        rani = random.randint(0, len(actual_data[1]) - 1)
        apoint = (actual_data[0][rani], actual_data[1][rani], actual_data[2][rani])

        lastk = (10000, 10000, 10000)
        for j in range(len(kax)):
            curk = (kalman_data[0][j], kalman_data[1][j], kalman_data[2][j])
            
            cur_distance = np.sqrt((curk[0] - apoint[0])**2 + (curk[1] - apoint[1])**2 + (curk[2] - apoint[2])**2)
            old_distance = np.sqrt((lastk[0] - apoint[0])**2 + (lastk[1] - apoint[1])**2 + (lastk[2] - apoint[2])**2)
            
            if cur_distance < old_distance:
                lastk = curk
        
        diff = abs(lastk[ax_idx] - apoint[ax_idx])
        avg_diff += diff

        p1 = apoint
        p2 = lastk

    if ret_type == "print":
        print(f'Average diff searching in {axes[ax_idx]} = {avg_diff / 100}')
        return p1, p2
    else:
        return avg_diff / 100


In [9]:
# Load layer configuration and sample event row data
layer_df = pd.read_csv('csv/layer_radius.csv')

def plot_fit(track):
    fig, ax = plt.subplots(2, 2, figsize=(14, 14), subplot_kw=dict(projection='3d'))
    
    kalman_data = [[], [], []]
    actual_data = [[], [], []]

    particle = particles[track]

    for hit in particle.kalman_hits:
        kalman_data[0].append(hit.hit_pos[0])
        kalman_data[1].append(hit.hit_pos[1])
        kalman_data[2].append(hit.hit_pos[2])

    for hit in particle.actual_hits:
        actual_data[0].append(hit.hit_pos[0])
        actual_data[1].append(hit.hit_pos[1])
        actual_data[2].append(hit.hit_pos[2])

    p1, p2 = get_accuracy(kalman_data, actual_data, 0, "print")
    p3, p4 = get_accuracy(kalman_data, actual_data, 1, "print")
    p5, p6 = get_accuracy(kalman_data, actual_data, 2, "print")
    
    # Plot fitted and actual tracks
    for i in range(2):
        for j in range(2):
            ax[i][j].plot(kalman_data[0], kalman_data[1], kalman_data[2], color="red", label="Kalman fitted")
            ax[i][j].plot(actual_data[0], actual_data[1], actual_data[2], color="black", label="Actual")

            ax[i][j].scatter([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color="blue", label="x test")
            ax[i][j].scatter([p3[0], p4[0]], [p3[1], p4[1]], [p3[2], p4[2]], color="green", label="y test")
            ax[i][j].scatter([p5[0], p6[0]], [p5[1], p6[1]], [p5[2], p6[2]], color="yellow", label="z test")

            ax[i][j].set_xlabel("x")
            ax[i][j].set_ylabel("y")
            ax[i][j].set_zlabel("z")

    ax[0][1].view_init(elev=90, azim=-90)
    ax[1][0].view_init(elev=180, azim=-90)
    ax[1][1].view_init(elev=180, azim=-180)

    handles, labels = ax[0][1].get_legend_handles_labels()

    for i in range(2):
        for j in range(2):
            ax[i][j].legend()
            
    plt.show()

In [31]:
#plot_fit(1)

In [11]:
def check_good_fits():
    num_particles = 0
    good_count = 0

    good_in_xy = 0
    
    for i in range(len(particles)):
        kalman_data = [[], [], []]
        actual_data = [[], [], []]

        particle = particles[i]

        for hit in particle.kalman_hits:
            kalman_data[0].append(hit.hit_pos[0])
            kalman_data[1].append(hit.hit_pos[1])
            kalman_data[2].append(hit.hit_pos[2])

        for hit in particle.actual_hits:
            actual_data[0].append(hit.hit_pos[0])
            actual_data[1].append(hit.hit_pos[1])
            actual_data[2].append(hit.hit_pos[2])

        if not len(actual_data[0]): continue
        
        acx = get_accuracy(kalman_data, actual_data, 0, "return")
        acy = get_accuracy(kalman_data, actual_data, 1, "return")
        acz = get_accuracy(kalman_data, actual_data, 2, "return")

        if acx == -1 or acy == -1 or acz == -1: continue
        num_particles += 1

        if (acx + acy + acz) < 20:
            good_count += 1

        if (acx + acy) < 15:
            good_in_xy += 1

    print(f"Percent of good fits = {good_count/num_particles *100:.2f}%")  
    print(f"Percent of good fits in xy = {good_in_xy/num_particles *100:2f}%")  


In [12]:
#check_good_fits()

## Signal

In [14]:
def plot_signal(m, G, tau):
    # Detector resolution jitter from diffusion values
    quad_sigma = np.std(np.sqrt(clusters_df["long_diffusion"]**2 + clusters_df["trans_diffusion"]**2), ddof=1)
    
    wire_times = []
    gain_list = []
    
    for time in clusters_df["hit_time"]:
        sampled_gain = np.random.gamma(shape=m, scale=G/m)
        gain_list.append(sampled_gain)
        
        quad_ran = random.gauss(0, quad_sigma)
        response_delay = np.random.gamma(shape=2, scale=tau)
        smeared_time = time + response_delay + quad_ran
        wire_times.append(smeared_time)

    # Order the data by increasing time
    sorted_pairs = sorted(zip(wire_times, gain_list))
    wire_times_sorted, gain_sorted = map(list, zip(*sorted_pairs))

    # Make all y values positive
    flipped_gain = abs((np.array(gain_sorted)-G))

    # Insert a value between each value in a list
    def insert_midpoints(lst):
        result = []
        for i in range(len(lst) - 1):
            result.append(lst[i])
            result.append((lst[i] + lst[i+1]) / 2)
        result.append(lst[-1])
        return result

    # Add a value with 0 gain after every point for clarity
    zeroed_times = insert_midpoints(wire_times_sorted[:150])
    zeroed_gains = [x for item in flipped_gain[:150] for x in (item, 0)]

    # Plot the signal function
    plt.figure(figsize=(12, 10))
    plt.plot(zeroed_times[:150], zeroed_gains[:150])
    plt.xlabel("Time [ns]")
    plt.ylabel("Charge/Gain")
    plt.show()

In [35]:
m = 10
G = 20000
tau = 8

#plot_signal(m, G, tau)

## Momentum resolution

In [17]:
def plot_momentum():
    mom_kalman = []
    mom_actual = []

    max_id = 999

    for i in range(max_id):
        cur_lowest = np.inf
        particle = particles[i]

        for hit in particle.kalman_hits:
            mom_mag = np.sqrt(hit.hit_mom[0]**2 + hit.hit_mom[1]**2)
            if mom_mag < cur_lowest: cur_lowest = mom_mag

        if cur_lowest == np.inf: continue

        # Only xy components
        mom_kalman.append(cur_lowest)
        mom_actual.append(particle.init_mom[0]**2 + particle.init_mom[1]**2)

    mom_actual = np.array(mom_actual)
    mom_kalman = np.array(mom_kalman)

    rel_unc = (mom_kalman - mom_actual)/mom_actual

    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, mom_kalman)
    plt.xlabel("Actual [MeV]")
    plt.ylabel("Kalman [MeV]")
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, (mom_kalman - mom_actual))
    plt.xlabel("Actual [MeV]")
    plt.ylabel("Kalman - Actual")
    plt.show()
    
    plt.figure(figsize=(10, 6))
    plt.scatter(mom_actual, rel_unc)
    plt.xlabel("Actual [MeV]")
    plt.ylabel("(Kalman - Actual) / Actual")
    plt.show()

In [39]:
#plot_momentum()